Copyright 2026 Google LLC.

SPDX-License-Identifier: Apache-2.0


<a href="https://colab.research.google.com/github/datacommonsorg/api-python/blob/master/notebooks/analyzing_income_distribution_sdmx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Analyzing Income Distribution with Data Commons SDMX

The American Community Survey (published by the US Census) annually reports the number of households in a given income bracket at the state level. We can use this information, stored in Data Commons, to visualize disparity in income for each state in the US. Our goal for this tutorial is to generate a plot that visualizes the total number of households across a given set of income brackets for a selected state.

Before we begin, we'll set up our notebook.

## Set up environment

### Install libraries

Install pySDMX for SDMX metadata and the Data Commons Python client for place names.


In [ ]:
!pip install "pysdmx[data,json]==1.19.0" "datacommons-client==2.1.6" --quiet


### Configure the API key

Obtain your own Data Commons API key from [apikeys.datacommons.org](https://apikeys.datacommons.org/) and replace the placeholder in the cell below with your key.

The notebook sends this key in the `X-Api-Key` header for SDMX requests and configures the Data Commons Python client with the same key. After changing the key, rerun this cell and the data cells below; the installation, imports, and helper definitions do not need to be rerun.


In [ ]:
from datacommons_client import DataCommonsClient

DC_API_KEY = "Replace this string with your API key"
dc_client = DataCommonsClient(api_key=DC_API_KEY)


### Import dependencies

Import the libraries and define the helpers required for data manipulation, analysis, and plotting.


In [ ]:
import matplotlib.pyplot as plt

import io
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from pysdmx.io import read_sdmx

API_ROOT = "https://api.datacommons.org"
SDMX_DATA_URL = f"{API_ROOT}/sdmx/v3/data/dataflow/DC/DF_OBS/1.0.0/*"
SDMX_AVAILABILITY_URL = (
    f"{API_ROOT}/sdmx/v3/availability/dataflow/DC/DF_OBS/1.0.0/*"
)


def _headers(accept=None):
    if not DC_API_KEY:
        raise ValueError("Set DC_API_KEY before requesting Data Commons.")
    headers = {"X-Api-Key": DC_API_KEY}
    if accept:
        headers["Accept"] = accept
    return headers


def sdmx_data(constraints):
    response = requests.get(
        SDMX_DATA_URL,
        params=constraints,
        headers=_headers("application/vnd.sdmx.data+csv;version=2.0.0"),
        timeout=120,
    )
    response.raise_for_status()
    frame = pd.read_csv(io.BytesIO(response.content), low_memory=False)
    frame["OBS_VALUE"] = pd.to_numeric(frame["OBS_VALUE"], errors="coerce")
    return frame


def availability_values(component, constraints):
    response = requests.get(
        f"{SDMX_AVAILABILITY_URL}/{component}",
        params=constraints,
        headers=_headers(),
        timeout=120,
    )
    response.raise_for_status()
    message = read_sdmx(io.BytesIO(response.content))
    if not message.structures:
        raise ValueError("The SDMX Availability request returned no constraint.")
    return {
        value.value
        for region in message.structures[0].cube_regions
        for key_value in region.key_values
        if key_value.id == component
        for value in key_value.values
    }


def require_variables(label, variables, constraints):
    constraints = {
        key: value for key, value in constraints.items()
        if not (key == "c[TIME_PERIOD]" and value == "LATEST")
    }
    available = availability_values(
        "variableMeasured",
        {**constraints, "c[variableMeasured]": ",".join(variables)},
    )
    missing = sorted(set(variables) - available)
    if missing:
        raise ValueError(f"{label} is missing variables: {', '.join(missing)}")
    print(f"{label}: all {len(variables)} variables are available.")


# The client follows response pagination within each input batch.
def _fetch_name_batch(nodes):
    return {
        dcid: name.value
        for dcid, name in dc_client.node.fetch_entity_names(
            entity_dcids=nodes
        ).items()
    }


def fetch_node_names(dcids, batch_size=500):
    unique_dcids = list(dict.fromkeys(dcids))
    if not unique_dcids:
        return {}
    batches = [
        unique_dcids[index:index + batch_size]
        for index in range(0, len(unique_dcids), batch_size)
    ]
    names = {dcid: dcid for dcid in unique_dcids}
    with ThreadPoolExecutor(max_workers=min(8, len(batches))) as executor:
        for batch_names in executor.map(_fetch_name_batch, batches):
            names.update(batch_names)
    return names


## 1. Getting the data

The Data Commons graph identifies 16 different income brackets. The list of these variables can be found under the **Economy > Household income** category in the [Statistical Variable Explorer](https://datacommons.org/tools/statvar).

SDMX Availability confirms that all 16 variables occur for US states under the Census ACS 5-year facet. It does not guarantee that every state has every bracket. SDMX Data then returns the latest values as SDMX-CSV, which Pandas reads into a DataFrame and filters to states containing all 16 brackets. The Data Commons Python client supplies names for the state DCIDs returned by SDMX.


In [ ]:
income_brackets = [
    "Count_Household_IncomeOfUpto10000USDollar",
    "Count_Household_IncomeOf10000To14999USDollar",
    "Count_Household_IncomeOf15000To19999USDollar",
    "Count_Household_IncomeOf20000To24999USDollar",
    "Count_Household_IncomeOf25000To29999USDollar",
    "Count_Household_IncomeOf30000To34999USDollar",
    "Count_Household_IncomeOf35000To39999USDollar",
    "Count_Household_IncomeOf40000To44999USDollar",
    "Count_Household_IncomeOf45000To49999USDollar",
    "Count_Household_IncomeOf50000To59999USDollar",
    "Count_Household_IncomeOf60000To74999USDollar",
    "Count_Household_IncomeOf75000To99999USDollar",
    "Count_Household_IncomeOf100000To124999USDollar",
    "Count_Household_IncomeOf125000To149999USDollar",
    "Count_Household_IncomeOf150000To199999USDollar",
    "Count_Household_IncomeOf200000OrMoreUSDollar",
]

income_constraints = {
    "c[observationAbout.containedInPlace+]": "country/USA",
    "c[observationAbout.typeOf]": "State",
    "c[provenance]": "dc/base/CensusACS5YearSurvey",
    "c[measurementMethod]": "CensusACS5yrSurvey",
    "c[TIME_PERIOD]": "LATEST",
}

require_variables("US state income", income_brackets, income_constraints)
income_rows = sdmx_data({
    **income_constraints,
    "c[variableMeasured]": ",".join(income_brackets),
}).dropna(subset=["OBS_VALUE"])

# Availability checks the slice as a whole; this verifies completeness per state.
complete = (
    income_rows.groupby("observationAbout")["variableMeasured"].nunique()
    == len(income_brackets)
)
income_rows = income_rows[
    income_rows["observationAbout"].isin(complete[complete].index)
]
names = fetch_node_names(income_rows["observationAbout"].unique())

data = (
    income_rows.rename(columns={
        "observationAbout": "entity",
        "variableMeasured": "variable",
        "OBS_VALUE": "value",
    })
    .assign(entity_name=lambda frame: frame["entity"].map(names))
    [["entity", "entity_name", "variable", "value"]]
)
print(f"Complete states: {data['entity'].nunique()}")
data.head()


## 2. Analyzing the data

Let's plot our data as a histogram. Notice that the income ranges as tabulated by the US Census are not equal. At the low end, the range is 0–9,999, whereas toward the top, the range 150,000–199,999 is five times as broad. We make the width of each column correspond to its range, which gives us an idea of total earnings, not just the number of households in that group.

First we provide code for generating the plot.


In [ ]:
label_to_range = {
    "Count_Household_IncomeOfUpto10000USDollar": [0, 9999],
    "Count_Household_IncomeOf10000To14999USDollar": [10000, 14999],
    "Count_Household_IncomeOf15000To19999USDollar": [15000, 19999],
    "Count_Household_IncomeOf20000To24999USDollar": [20000, 24999],
    "Count_Household_IncomeOf25000To29999USDollar": [25000, 29999],
    "Count_Household_IncomeOf30000To34999USDollar": [30000, 34999],
    "Count_Household_IncomeOf35000To39999USDollar": [35000, 39999],
    "Count_Household_IncomeOf40000To44999USDollar": [40000, 44999],
    "Count_Household_IncomeOf45000To49999USDollar": [45000, 49999],
    "Count_Household_IncomeOf50000To59999USDollar": [50000, 59999],
    "Count_Household_IncomeOf60000To74999USDollar": [60000, 74999],
    "Count_Household_IncomeOf75000To99999USDollar": [75000, 99999],
    "Count_Household_IncomeOf100000To124999USDollar": [100000, 124999],
    "Count_Household_IncomeOf125000To149999USDollar": [125000, 149999],
    "Count_Household_IncomeOf150000To199999USDollar": [150000, 199999],
    "Count_Household_IncomeOf200000OrMoreUSDollar": [200000, 300000],
}


def plot_income(frame, state_name):
    selected = frame.loc[frame["entity_name"] == state_name]
    if selected.empty:
        print(f"{state_name} does not have sufficient income data.")
        return None

    values = (
        selected.groupby("variable")["value"].first().reindex(income_brackets)
    )
    if values.isna().any():
        print(f"{state_name} does not have all income brackets.")
        return None

    widths_without_interval = [
        int((label_to_range[bracket][1] - label_to_range[bracket][0]) / 18)
        for bracket in income_brackets
    ]
    positions = []
    total = 0
    for width in widths_without_interval:
        positions.append(total + width // 2)
        total += width
    widths = [width - 50 for width in widths_without_interval]

    plt.figure(figsize=(12, 10))
    plt.xticks(positions, income_brackets, rotation=90)
    plt.grid(True)
    plt.bar(positions, values.values, widths, color="b", alpha=0.3)
    plt.title(f"Household income distribution: {state_name}")
    return selected


We can then call this code with a state to plot the income bracket sizes. Changing the state uses data already held in the DataFrame and does not make another API request.


In [ ]:
#@title Enter state to plot { run: "auto" }
state_name = "Washington" #@param ["Missouri", "Arkansas", "Arizona", "Ohio", "Connecticut", "Vermont", "Illinois", "South Dakota", "Iowa", "Oklahoma", "Kansas", "Washington", "Oregon", "Hawaii", "Minnesota", "Idaho", "Alaska", "Colorado", "Delaware", "Alabama", "North Dakota", "Michigan", "California", "Indiana", "Kentucky", "Nebraska", "Louisiana", "New Jersey", "Rhode Island", "Utah", "Nevada", "South Carolina", "Wisconsin", "New York", "North Carolina", "New Hampshire", "Georgia", "Pennsylvania", "West Virginia", "Maine", "Mississippi", "Montana", "Tennessee", "New Mexico", "Massachusetts", "Wyoming", "Maryland", "Florida", "Texas", "Virginia"]
result = plot_income(data, state_name)
plt.show()


We can also display the raw table of values.


In [ ]:
result


This is only the beginning! What else can you analyze? For example, you could try computing a measure of income disparity in each state (see [Gini coefficient](https://en.wikipedia.org/wiki/Gini_coefficient)).

You could then expand the DataFrame to include more information and analyze how attributes such as education level, crime, or even weather affect income disparity.
